# 04 — Simple ML baselines

**Goal.** Find out how much of the signal each *representation* carries, using ordinary
classifiers and an honest evaluation protocol.

**The question is about representations, not algorithms.** We compare:

| representation | what it is |
|---|---|
| majority class | predict "negative" always — the floor |
| transcript length | one number: how many words the person said |
| surface statistics | seven cheap numbers (length, vocabulary, sentence counts) |
| LLM features v1 | the ten categorical features from notebook 02 |
| surface + LLM features | both together |
| TF-IDF word n-grams | plain bag-of-words |
| TF-IDF character n-grams | character 3–5 grams — the right choice for a highly inflected language like Czech |

**Protocol.** Repeated stratified 5-fold cross-validation on the 241 training documents.
Repeated, because with 241 documents a single 5-fold split is noisy; stratified, because only
29% of documents are positive. The test set is not used anywhere in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_REPEATS = 10

corpus = pd.read_csv("outputs/corpus.csv")
features = pd.read_csv("outputs/features_v1_train.csv")
FEATURE_COLS = [c for c in features.columns if c not in ("File", "Class", "raw_llm_output")]
SURFACE = ["n_word", "n_type", "ttr", "n_sent", "mlu", "n_comma", "n_char"]

train = corpus[corpus.split == "train"].merge(
    features[["File"] + FEATURE_COLS], left_on="file", right_on="File", validate="1:1"
).reset_index(drop=True)
y = train.label.astype(int).values
print(f"training on {len(train)} documents ({y.sum()} positive) — test set not loaded")

training on 241 documents (70 positive) — test set not loaded


## The evaluation function

Four metrics, because each answers a different question:

* **AUC** — can the model rank a positive case above a negative one? Insensitive to class
  imbalance and to where we set the threshold. This is the main number.
* **Balanced accuracy** — average of sensitivity and specificity. Unlike plain accuracy, a
  majority-class predictor scores 0.5 here.
* **Macro-F1** — precision/recall balance treating both classes as equally important.
* **Accuracy** — reported because people expect it, but it is misleading at 71/29 imbalance.

We average out-of-fold predicted probabilities over `N_REPEATS` different fold assignments,
then compute the metrics once on those averaged predictions. `± ` shows the spread across repeats.

In [2]:
def evaluate(name, model, X, y, n_repeats=N_REPEATS):
    """Repeated stratified 5-fold CV. Returns one row of results."""
    probs = np.zeros((n_repeats, len(y)))
    per_repeat_auc = []
    for r in range(n_repeats):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + r)
        p = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
        probs[r] = p
        per_repeat_auc.append(roc_auc_score(y, p))

    p_mean = probs.mean(axis=0)
    pred = (p_mean >= 0.5).astype(int)
    row = {
        "representation": name,
        "auc": roc_auc_score(y, p_mean),
        "auc_sd": float(np.std(per_repeat_auc)),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "f1_positive": f1_score(y, pred),
        "accuracy": accuracy_score(y, pred),
    }
    print(f"{name:28} AUC {row['auc']:.3f}±{row['auc_sd']:.3f}   "
          f"BalAcc {row['balanced_accuracy']:.3f}   macroF1 {row['macro_f1']:.3f}   "
          f"Acc {row['accuracy']:.3f}")
    return row, p_mean

## Building blocks

`class_weight="balanced"` tells the classifier to weight the 70 positive documents as heavily
as the 171 negative ones. Without it, most models simply learn to say "negative".

In [3]:
def logreg():
    return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)

def onehot():
    # min_frequency=5 folds very rare feature levels together instead of giving each its own column
    return OneHotEncoder(handle_unknown="ignore", min_frequency=5)

def tfidf_word():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)

def tfidf_char():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True)

results, probs = [], {}

## Baseline 0 — the floor

Whatever we build has to beat this. Note the accuracy: 0.71 without looking at the text at all.

In [4]:
row, p = evaluate("Majority class", DummyClassifier(strategy="most_frequent"),
                  train[["n_word"]], y, n_repeats=1)
results.append(row); probs["Majority class"] = p

Majority class               AUC 0.500±0.000   BalAcc 0.500   macroF1 0.415   Acc 0.710


## Baselines 1–2 — how far does counting words get you?

In [5]:
for name, cols in [("Transcript length only", ["n_word"]), ("Surface statistics", SURFACE)]:
    row, p = evaluate(name, make_pipeline(StandardScaler(), logreg()), train[cols], y)
    results.append(row); probs[name] = p

Transcript length only       AUC 0.693±0.002   BalAcc 0.639   macroF1 0.606   Acc 0.631


Surface statistics           AUC 0.742±0.009   BalAcc 0.672   macroF1 0.644   Acc 0.672


## Baseline 3 — the LLM features

Three classifiers on the same representation, to check that a weak result is a property of the
features and not of the model choice.

In [6]:
# The thesis protocol (see AGENT_CONTEXT s6/s10) caps the classifier family to
# ElasticNet / L2 logistic regression / linear SVM. This exploratory V1 baseline
# therefore reports plain L2 logistic regression only -- the earlier random-forest
# and XGBoost arms were dropped to keep the comparison inside that cap.
row, p = evaluate("LLM features (logreg)", make_pipeline(onehot(), logreg()), train[FEATURE_COLS], y)
results.append(row); probs["LLM features (logreg)"] = p


LLM features (logreg)        AUC 0.694±0.013   BalAcc 0.632   macroF1 0.620   Acc 0.664


Compare the three. If the flexible tree models do *worse* than logistic regression, that is
the overfitting notebook 03 predicted: with 241 documents and nearly as many distinct feature
combinations, a tree can memorise rather than generalise.

## Baseline 4 — surface and LLM features together

In [7]:
row, p = evaluate(
    "Surface + LLM features",
    make_pipeline(
        ColumnTransformer([("num", StandardScaler(), SURFACE), ("cat", onehot(), FEATURE_COLS)]),
        logreg(),
    ),
    pd.concat([train[SURFACE], train[FEATURE_COLS]], axis=1), y,
)
results.append(row); probs["Surface + LLM features"] = p

Surface + LLM features       AUC 0.757±0.010   BalAcc 0.703   macroF1 0.693   Acc 0.734


## Baseline 5 — plain lexical models

No LLM involved: just term frequencies. Character n-grams matter for Czech because the same
word appears in many inflected forms, which a word-level model treats as unrelated tokens.

In [8]:
for name, vec in [("TF-IDF word 1-2gram", tfidf_word()), ("TF-IDF char 3-5gram", tfidf_char())]:
    row, p = evaluate(name, make_pipeline(vec, logreg()), train.text, y)
    results.append(row); probs[name] = p

TF-IDF word 1-2gram          AUC 0.842±0.009   BalAcc 0.752   macroF1 0.760   Acc 0.809


TF-IDF char 3-5gram          AUC 0.868±0.009   BalAcc 0.753   macroF1 0.758   Acc 0.805


## All results

In [9]:
res = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
res.to_csv("outputs/baseline_results.csv", index=False)
pd.DataFrame(probs).assign(label=y).to_csv("outputs/baseline_oof_probs.csv", index=False)

        representation   auc  auc_sd  balanced_accuracy  macro_f1  f1_positive  accuracy
   TF-IDF char 3-5gram 0.868   0.009              0.753     0.758        0.652     0.805
   TF-IDF word 1-2gram 0.842   0.009              0.752     0.760        0.652     0.809
Surface + LLM features 0.757   0.010              0.703     0.693        0.579     0.734
    Surface statistics 0.742   0.009              0.672     0.644        0.543     0.672
 LLM features (logreg) 0.694   0.013              0.632     0.620        0.491     0.664
Transcript length only 0.693   0.002              0.639     0.606        0.508     0.631
        Majority class 0.500   0.000              0.500     0.415        0.000     0.710


## What is the lexical model picking up on?

If TF-IDF does well, we should know why — both because it is interesting linguistically and
because it tells us what a good LLM feature ought to measure.

In [10]:
vec = TfidfVectorizer(min_df=3, sublinear_tf=True)
X = vec.fit_transform(train.text)
clf = logreg().fit(X, y)
words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])

print("words most associated with the POSITIVE class (MCI / dementia):")
print(" ", ", ".join(words[order[-25:]][::-1]))
print("\nwords most associated with the NEGATIVE class (cognitively normal):")
print(" ", ", ".join(words[order[:25]]))

words most associated with the POSITIVE class (MCI / dementia):
  veverka, leze, nevim, tady, jídlem, to, co, plavec, všecko, pejsek, chce, pani, pití, já, snad, deštníkem, čtení, sem, no, deštník, hrajou, holub, děcka, kačenky, nebo

words most associated with the NEGATIVE class (cognitively normal):
  sedí, pták, piknik, padá, letí, skáče, břehu, muž, knihu, na, vody, žába, plave, do, obloze, vykukuje, ptáků, vlevo, hraní, starší, hází, plují, dítě, pozadí, má


### Summary

* Compare each representation against **transcript length only**. Anything that does not beat
  a word count is not yet contributing linguistic information.
* Compare the three classifiers on the LLM features. Similar or worse performance from the
  flexible models is evidence about the representation, not about the algorithm.
* The word lists above are the most useful output of this notebook for the thesis: they name
  the linguistic constructs that actually separate the groups, and they are what the v2
  feature design in notebook 06 is built from.